<a href="https://colab.research.google.com/github/BasamYashwanth/Deep-Learning/blob/main/DL_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ---------------------------------------------------------
# Build and Train a ConvNet in PyTorch for Classification
# Dataset: CIFAR-10
# ---------------------------------------------------------

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms


# ---------------------------------------------------------
# 1. Select device
# ---------------------------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using device:", device)


# ---------------------------------------------------------
# 2. Define transformations
# ---------------------------------------------------------

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.5, 0.5, 0.5),
        std=(0.5, 0.5, 0.5)
    )
])


# ---------------------------------------------------------
# 3. Load CIFAR-10 dataset
# ---------------------------------------------------------

train_dataset = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform
)

test_dataset = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform
)


# ---------------------------------------------------------
# 4. Create DataLoaders
# ---------------------------------------------------------

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False
)


# ---------------------------------------------------------
# 5. Define the ConvNet
# ---------------------------------------------------------

class ConvNet(nn.Module):

    def __init__(self):
        super(ConvNet, self).__init__()

        # Convolutional Block 1
        self.conv1 = nn.Conv2d(
            in_channels=3,
            out_channels=32,
            kernel_size=3,
            padding=1
        )

        self.pool = nn.MaxPool2d(
            kernel_size=2,
            stride=2
        )

        # Convolutional Block 2
        self.conv2 = nn.Conv2d(
            in_channels=32,
            out_channels=64,
            kernel_size=3,
            padding=1
        )

        # Fully connected layers
        self.fc1 = nn.Linear(
            64 * 8 * 8,
            128
        )

        self.fc2 = nn.Linear(
            128,
            10
        )

        self.relu = nn.ReLU()

    def forward(self, x):

        # First convolution block
        x = self.relu(self.conv1(x))
        x = self.pool(x)

        # Second convolution block
        x = self.relu(self.conv2(x))
        x = self.pool(x)

        # Flatten feature maps
        x = x.view(x.size(0), -1)

        # Fully connected layer
        x = self.relu(self.fc1(x))

        # Output layer
        x = self.fc2(x)

        return x


# ---------------------------------------------------------
# 6. Create model
# ---------------------------------------------------------

model = ConvNet().to(device)

print("\nModel Architecture:")
print(model)


# ---------------------------------------------------------
# 7. Define loss function and optimizer
# ---------------------------------------------------------

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)


# ---------------------------------------------------------
# 8. Train the ConvNet
# ---------------------------------------------------------

num_epochs = 5

for epoch in range(num_epochs):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        # Move data to CPU/GPU
        images = images.to(device)
        labels = labels.to(device)

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)

        # Calculate loss
        loss = criterion(outputs, labels)

        # Backward pass
        loss.backward()

        # Update weights
        optimizer.step()

        # Calculate statistics
        running_loss += loss.item()

        _, predicted = torch.max(outputs.data, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total

    print(
        f"Epoch [{epoch + 1}/{num_epochs}] "
        f"Loss: {running_loss / len(train_loader):.4f} "
        f"Accuracy: {accuracy:.2f}%"
    )


# ---------------------------------------------------------
# 9. Evaluate the model
# ---------------------------------------------------------

model.eval()

correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs.data, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()


test_accuracy = 100 * correct / total

print("\nTest Accuracy: {:.2f}%".format(test_accuracy))


# ---------------------------------------------------------
# 10. Test prediction for one batch
# ---------------------------------------------------------

classes = (
    "plane",
    "car",
    "bird",
    "cat",
    "deer",
    "dog",
    "frog",
    "horse",
    "ship",
    "truck"
)

images, labels = next(iter(test_loader))

images = images.to(device)

with torch.no_grad():
    outputs = model(images)

_, predictions = torch.max(outputs, 1)

print("\nSample Predictions:")

for i in range(10):
    print(
        f"Actual: {classes[labels[i]]:>5} | "
        f"Predicted: {classes[predictions[i]]:>5}"
    )

Using device: cpu


100%|██████████| 170M/170M [41:36<00:00, 68.3kB/s]



Model Architecture:
ConvNet(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (fc1): Linear(in_features=4096, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=10, bias=True)
  (relu): ReLU()
)
Epoch [1/5] Loss: 1.3407 Accuracy: 51.92%
Epoch [2/5] Loss: 0.9743 Accuracy: 65.50%
Epoch [3/5] Loss: 0.8167 Accuracy: 71.44%
Epoch [4/5] Loss: 0.6987 Accuracy: 75.40%
Epoch [5/5] Loss: 0.5994 Accuracy: 79.04%

Test Accuracy: 71.75%

Sample Predictions:
Actual:   cat | Predicted:   cat
Actual:  ship | Predicted:  ship
Actual:  ship | Predicted:  ship
Actual: plane | Predicted:  ship
Actual:  frog | Predicted:  frog
Actual:  frog | Predicted:  frog
Actual:   car | Predicted:   car
Actual:  frog | Predicted:  frog
Actual:   cat | Predicted:   cat
Actual:   car | Predicted:   car
